# Kenzan: VLM on held-out v3 (Colab)

Runtime → Change runtime type → **T4 GPU**. Default: Qwen3-VL-8B-Instruct, 4-bit weights, **fp16 compute** (the T4 has no native bf16).

Held-out v3 is **8 new degradations of the same reference page**, not new documents.

This notebook only produces raw model responses (`vlm_cache/`); scoring is offline (`stress/run_vlm_compare.py`). Run the cells in order. **Step 3 is a smoke test on the clean reference page; if it reads fewer than 70 of 74 cells correctly, stop and send `vlm_cache.zip` (step 5) — do not run step 4.**

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!git clone -q https://github.com/vedantjhunjhunwal/Kenzan.git
%cd Kenzan
# pinned: the VLM run uses the frozen commit fcaa41d (pushed before any VLM output existed)
!git checkout -q fcaa41d
!git rev-parse HEAD
!pip install -q -r requirements-vlm.txt
import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)

## 2. Held-out pages
Regenerate the 8 pages and check their image hashes. If any line says MISMATCH, upload `heldout_v3_pdfs.zip` with the file browser and run the commented unzip line instead.

In [ ]:
!python stress/make_pdfs.py stress/specs/heldout_v3.json stress/out_heldout_v3/pdfs
# only if MISMATCH:
# !unzip -o ../heldout_v3_pdfs.zip -d stress/out_heldout_v3/pdfs

## 3. Smoke test (clean reference page at 300 dpi; not a held-out page)
Loads the model (~5 min download) and reads one page. Pass = at least 70 of 74 cells correct.

In [ ]:
!python scripts/vlm_run.py --smoke-only --cache vlm_cache --quant 4bit; echo exit code $?
import json; sm = json.load(open('vlm_cache/smoke/smoke_result.json'))
print('SMOKE', sm['cells_correct'], '/', sm['cells_gt'], 'passed' if sm['passed'] else 'FAILED: STOP HERE, go to step 5')

## 4. Held-out v3 (only if the smoke test passed)
The script repeats the smoke test from cache and refuses to continue if it failed.

In [ ]:
assert json.load(open('vlm_cache/smoke/smoke_result.json'))['passed'], 'smoke test failed: do not run v3'
!python scripts/vlm_run.py --pdfs stress/out_heldout_v3/pdfs --cache vlm_cache --quant 4bit

## 5. Download

In [ ]:
!zip -qr vlm_cache.zip vlm_cache
from google.colab import files
files.download('vlm_cache.zip')